# Checkout Methods Analysis Lab

Additional product analytics lab notebook included as a reviewable portfolio artifact.

This public portfolio copy keeps the full notebook source visible on GitHub while removing execution outputs, execution counts, and environment-specific metadata.

# Session 2.3 — Payment Methods at Checkout: Analysis Exercise
### MSc Data Science · Payment Analysis for Open Banking

You have been given a dataset of **14,000+ checkout sessions** across 60 days from 5 merchants.

Each row is one checkout attempt. The dataset covers four payment methods:

| Method | Description |
|--------|-------------|
| `CARD` | Credit/debit card via 3DS 2.0 |
| `PAY_BY_BANK` | Open Banking Payment Initiation (PIS) |
| `PAYPAL` | PayPal wallet |
| `APPLE_PAY` | Apple Pay / Google Pay (mobile-native) |


---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

COLORS = {
    'CARD':        '#163D6E',
    'PAY_BY_BANK': '#0D9488',
    'PAYPAL':      '#6366F1',
    'APPLE_PAY':   '#1C1C1E',
}

df = pd.read_csv('Lesson 2 -checkout_sessions.csv', parse_dates=['timestamp'])
df['date'] = pd.to_datetime(df['date'])

print(f"Rows: {len(df):,}")
print(f"Methods: {df['payment_method'].unique()}")
print(f"Merchants: {df['merchant_name'].unique()}")
print()
print(df.dtypes)

After running the cell above and authenticating with Google Drive, your files will be accessible under `/content/drive/MyDrive/`. You will need to update the path to your CSV file accordingly. For example, if your file is in a folder named `your_shared_folder` inside your Drive, the path might look like `/content/drive/MyDrive/your_shared_folder/Lesson 2 -checkout_sessions.csv`.

---
## Task 1 — Conversion Rate by Payment Method

**Conversion rate** = sessions that completed payment / total sessions that started.

Calculate overall conversion rate per method, then break it down by:
- Device (mobile vs desktop)
- Basket value band (< £50 / £50–200 / > £200)

> **Think about:** What drives the gap between methods? Is it UX friction, SCA, redirects, or user familiarity?

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────
# 1a. Overall conversion rate per method


# 1b. Conversion by device


# 1c. Create basket bands and show conversion by method × band


#### ✅ Answer — Task 1

In [ ]:
# 1a. Overall conversion
conv_overall = (
    df.groupby('payment_method')['converted']
    .agg(['sum','count','mean'])
    .rename(columns={'sum':'conversions','count':'sessions','mean':'conv_rate'})
    .assign(conv_rate_pct=lambda d: (d['conv_rate']*100).round(1))
    .sort_values('conv_rate', ascending=False)
)
print("Overall conversion rates:")
print(conv_overall[['sessions','conversions','conv_rate_pct']])

# 1b. By device
conv_device = (
    df.groupby(['payment_method','device'])['converted']
    .mean().mul(100).round(1).unstack()
)
print("\nConversion by device (%):")
print(conv_device)

# 1c. By basket band
df['basket_band'] = pd.cut(
    df['basket_value_gbp'],
    bins=[0, 50, 200, 9999],
    labels=['< £50', '£50–200', '> £200']
)
conv_basket = (
    df.groupby(['payment_method','basket_band'])['converted']
    .mean().mul(100).round(1).unstack()
)
print("\nConversion by basket band (%):")
print(conv_basket)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

methods = conv_overall.index.tolist()
colors  = [COLORS[m] for m in methods]
axes[0].barh(methods, conv_overall['conv_rate_pct'], color=colors)
axes[0].set_xlabel('Conversion rate (%)')
axes[0].set_title('Overall Conversion Rate')
for i, v in enumerate(conv_overall['conv_rate_pct']):
    axes[0].text(v + 0.3, i, f'{v}%', va='center', fontsize=10)

conv_mobile_desktop = df[df['device'].isin(['mobile','desktop'])].groupby(
    ['payment_method','device'])['converted'].mean().mul(100).unstack()
conv_mobile_desktop.plot(kind='bar', ax=axes[1], color=['#0D9488','#163D6E'], width=0.6)
axes[1].set_title('Conversion: Mobile vs Desktop')
axes[1].set_ylabel('Conversion rate (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print("""
Key insight:
  - Apple Pay has by far the highest mobile conversion — biometric auth = 1 tap.
  - Pay by Bank drops 10+ pts on mobile due to bank app redirect friction.
  - Card conversion falls on high basket value (SCA step-up challenges).
  - PayPal is relatively flat across devices — stored credentials reduce friction.
""")

---
## Task 2 — Cost per Completed Transaction

Calculate the **effective merchant cost** for each payment method:
- Average fee in £ per completed transaction
- Average effective rate as % of basket value
- Total fees paid over the full 60-day dataset

Then calculate the **cost at different basket values** — build a function that computes fee for a given method and basket value, and plot the fee curves from £10 to £500.

> **Think about:** At what basket value does PayPal become more expensive than cards? At what value does Pay by Bank fixed fee become negligible?

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────
# 2a. Average cost per completed transaction


# 2b. Fee curve: define fee_for_method(method, basket) and plot for £10–£500


#### ✅ Answer — Task 2

In [ ]:
# 2a. Cost summary (converted only)
converted = df[df['converted'] == True]

cost_summary = (
    converted.groupby('payment_method')
    .agg(
        conversions     = ('session_id', 'count'),
        avg_basket      = ('basket_value_gbp', 'mean'),
        avg_fee_gbp     = ('fee_gbp', 'mean'),
        total_fees_gbp  = ('fee_gbp', 'sum'),
        avg_rate_pct    = ('effective_rate_pct', 'mean'),
        total_net_rev   = ('net_revenue_gbp', 'sum'),
    )
    .round(3)
    .sort_values('avg_rate_pct')
)
print("Cost per completed transaction:")
print(cost_summary)

# 2b. Fee curves
FEE_PARAMS = {
    'CARD':        {'pct': 0.0150, 'fixed': 0.20},
    'PAY_BY_BANK': {'pct': 0.0000, 'fixed': 0.08},
    'PAYPAL':      {'pct': 0.0249, 'fixed': 0.30},
    'APPLE_PAY':   {'pct': 0.0155, 'fixed': 0.20},
}

def fee_for_method(method, basket):
    p = FEE_PARAMS[method]
    return basket * p['pct'] + p['fixed']

baskets = np.linspace(10, 500, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Absolute fee
for method in FEE_PARAMS:
    fees = [fee_for_method(method, b) for b in baskets]
    axes[0].plot(baskets, fees, color=COLORS[method], label=method, linewidth=2)
axes[0].set_xlabel('Basket value (£)')
axes[0].set_ylabel('Fee (£)')
axes[0].set_title('Absolute Fee by Basket Value')
axes[0].legend()

# Effective rate %
for method in FEE_PARAMS:
    rates = [fee_for_method(method, b) / b * 100 for b in baskets]
    axes[1].plot(baskets, rates, color=COLORS[method], label=method, linewidth=2)
axes[1].set_xlabel('Basket value (£)')
axes[1].set_ylabel('Effective rate (%)')
axes[1].set_title('Effective Fee Rate by Basket Value')
axes[1].legend()
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].axhline(y=1.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.show()

# Crossover analysis
breakeven_paypal_vs_card = (0.30 - 0.20) / (0.0249 - 0.0150)
pbb_parity = 0.08 / 0.015  # at what basket does PBB fixed fee = 1.5%?
print(f"PayPal becomes cheaper than Card below: £{breakeven_paypal_vs_card:.0f}")
print(f"Pay by Bank fixed fee (£0.08) = 1.5% of basket at: £{pbb_parity:.0f}")
print("""
Key insight:
  - Pay by Bank has the lowest cost at every basket value > ~£5.
  - PayPal is cheapest below ~£10 (low fixed fee), most expensive above.
  - Card and Apple Pay are nearly identical in cost (Apple pays ~0.15% to card network).
  - The Pay by Bank fixed fee (£0.08) becomes negligible above ~£50.
""")

---
## Task 3 — Fraud & Chargeback Analysis

Calculate per payment method:
- Fraud rate (% of completed transactions)
- Chargeback rate
- Who bears the liability (merchant vs bank)
- Estimated annual fraud cost (scale up 60-day total to 365 days)

> **Think about:** PayPal chargebacks are 'buyer protection disputes' — not the same as card chargebacks. Why does liability model matter more than raw fraud rate?

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────


#### ✅ Answer — Task 3

In [ ]:
fraud_summary = (
    converted.groupby('payment_method')
    .agg(
        conversions   = ('session_id', 'count'),
        fraud_n       = ('is_fraud', 'sum'),
        chargeback_n  = ('is_chargeback', 'sum'),
        total_basket  = ('basket_value_gbp', 'sum'),
        liability     = ('liability', lambda x: x.mode()[0]),
    )
)

fraud_summary['fraud_rate_pct']      = (fraud_summary['fraud_n'] / fraud_summary['conversions'] * 100).round(3)
fraud_summary['chargeback_rate_pct'] = (fraud_summary['chargeback_n'] / fraud_summary['conversions'] * 100).round(3)
fraud_summary['combined_rate_pct']   = fraud_summary['fraud_rate_pct'] + fraud_summary['chargeback_rate_pct']

# Annualise
scale = 365 / 60
fraud_summary['est_annual_fraud_cost_gbp'] = (
    (fraud_summary['fraud_n'] + fraud_summary['chargeback_n'])
    / fraud_summary['conversions']
    * (fraud_summary['total_basket'] / fraud_summary['conversions'])  # avg basket
    * fraud_summary['conversions'] * scale
).round(0)

print(fraud_summary[['conversions','fraud_rate_pct','chargeback_rate_pct',
                      'combined_rate_pct','liability','est_annual_fraud_cost_gbp']].to_string())

fig, ax = plt.subplots(figsize=(9, 4))
methods = fraud_summary.index.tolist()
x = np.arange(len(methods))
w = 0.35
ax.bar(x - w/2, fraud_summary['fraud_rate_pct'], w, label='Fraud', color=[COLORS[m] for m in methods], alpha=0.9)
ax.bar(x + w/2, fraud_summary['chargeback_rate_pct'], w, label='Chargeback', color=[COLORS[m] for m in methods], alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel('Rate (%)')
ax.set_title('Fraud Rate vs Chargeback Rate by Payment Method')
ax.legend()
for i, m in enumerate(methods):
    lib = fraud_summary.loc[m, 'liability']
    ax.annotate(f"Liability:\n{lib}", (i, -0.18), ha='center', fontsize=8, color='gray')
plt.tight_layout()
plt.show()

print("""
Key insight:
  - PayPal has 3x the combined fraud+chargeback rate vs cards.
  - BUT: PayPal buyer protection disputes are the merchant's cost regardless of fraud outcome.
  - Pay by Bank has near-zero chargebacks — push payments can't be reversed by the sender.
  - Liability model matters: a Pay by Bank fraud is the BANK's problem; a card fraud is the MERCHANT's.
  - Apple Pay tokenisation dramatically reduces card fraud rate vs standard card.
""")

---
## Task 4 — Settlement Speed Analysis

For each payment method:
- Show the distribution of `settlement_days`
- Calculate total GBP 'in transit' at any point (average daily completed sales × settlement lag)
- For a merchant doing £50k/day, what is the working capital tied up by each method?

> **Think about:** Why does a £50k/day merchant care deeply about T+0 vs T+2? What's the cash flow implication over a year?

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────


#### ✅ Answer — Task 4

In [ ]:
settle_summary = (
    converted.groupby('payment_method')['settlement_days']
    .agg(['mean','min','max',
          lambda x: (x == 0).mean() * 100,
          lambda x: (x == 1).mean() * 100,
          lambda x: (x == 2).mean() * 100])
)
settle_summary.columns = ['avg_days','min_days','max_days','pct_T0','pct_T1','pct_T2']
print("Settlement distribution:")
print(settle_summary.round(1))

# Working capital impact
daily_revenue = 50_000
print("\nWorking capital tied up (£50k/day merchant):")
for method in settle_summary.index:
    avg = settle_summary.loc[method, 'avg_days']
    tied = daily_revenue * avg
    annual_cost = tied * 0.05  # opportunity cost at 5% p.a.
    print(f"  {method:15}: £{tied:>10,.0f} tied up  |  ~£{annual_cost:>6,.0f}/yr opportunity cost")

# Plot settlement distribution
fig, ax = plt.subplots(figsize=(9, 4))
methods_list = list(COLORS.keys())
day_labels   = ['T+0', 'T+1', 'T+2']
pct_cols     = ['pct_T0', 'pct_T1', 'pct_T2']
x = np.arange(len(day_labels))
w = 0.2
for i, method in enumerate(methods_list):
    if method in settle_summary.index:
        vals = [settle_summary.loc[method, c] for c in pct_cols]
        ax.bar(x + i*w - 0.3, vals, w, label=method, color=COLORS[method])
ax.set_xticks(x)
ax.set_xticklabels(day_labels)
ax.set_ylabel('% of transactions')
ax.set_title('Settlement Timing Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print("""
Key insight:
  - Pay by Bank settles T+0 via Faster Payments — zero working capital tied up.
  - Cards settle T+2 → a merchant doing £50k/day has £100k permanently in transit.
  - At 5% cost of capital that's ~£5,000/year per £50k daily volume — before early funding fees.
  - This is why high-volume merchants pay acquirers for 'next-day funding' products.
""")

---
## Task 5 — UX Friction Score

Analyse the relationship between checkout friction and conversion:
- Use `steps_to_complete`, `redirect_required`, `sca_required` as friction indicators
- Examine `abandon_stage` distribution — where do users drop off for each method?
- Calculate an abandonment funnel by stage for each method

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────


#### ✅ Answer — Task 5

In [ ]:
# Friction profile per method
friction = (
    df.groupby('payment_method')
    .agg(
        avg_steps   = ('steps_to_complete', 'mean'),
        redirect_pct= ('redirect_required', 'mean'),
        sca_pct     = ('sca_required', 'mean'),
        conv_rate   = ('converted', 'mean'),
    )
    .round(3)
)
friction['redirect_pct'] *= 100
friction['sca_pct'] *= 100
friction['conv_rate_pct'] = friction['conv_rate'] * 100
print(friction[['avg_steps','redirect_pct','sca_pct','conv_rate_pct']])

# Abandonment stages
abandoned = df[df['converted'] == False]
abandon_by_method = (
    abandoned.groupby(['payment_method','abandon_stage'])
    .size().unstack(fill_value=0)
)
# Normalise to % of total abandons per method
abandon_pct = abandon_by_method.div(abandon_by_method.sum(axis=1), axis=0) * 100
print("\nAbandonment stage breakdown (%):")
print(abandon_pct.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Steps vs conversion
for method in friction.index:
    axes[0].scatter(
        friction.loc[method, 'avg_steps'],
        friction.loc[method, 'conv_rate_pct'],
        color=COLORS[method], s=200, zorder=5, label=method
    )
    axes[0].annotate(method, (
        friction.loc[method,'avg_steps'] + 0.05,
        friction.loc[method,'conv_rate_pct']
    ), fontsize=9)
axes[0].set_xlabel('Avg steps to complete')
axes[0].set_ylabel('Conversion rate (%)')
axes[0].set_title('Steps vs Conversion Rate')

# Abandonment heatmap as bar
abandon_pct.plot(kind='bar', ax=axes[1], width=0.7)
axes[1].set_title('Where Users Abandon (%)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("""
Key insight:
  - Strong negative correlation between steps and conversion rate.
  - Pay by Bank has 35% of abandonments at 'redirect_timeout' — app switching on mobile kills UX.
  - Apple Pay abandons mostly at 'payment_details' (face ID failure) — rare but decisive.
  - PayPal abandons happen mostly at payment_details — users forget passwords.
  - SCA challenge abandonment for cards (~30%) is the most remediable via 3DS optimisation.
""")

---
## Task 6 — Merchant & Segment Analysis

The dataset contains 5 merchant types. Analyse whether the optimal payment method differs by merchant:
- Compare conversion and cost across merchant types
- Identify if B2B transactions (`is_b2b = True`) behave differently
- Which method performs best for high-basket travel transactions?

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────


#### ✅ Answer — Task 6

In [ ]:
# Conversion by merchant type × payment method
merchant_conv = (
    df.groupby(['merchant_type','payment_method'])['converted']
    .mean().mul(100).round(1).unstack()
)
print("Conversion by merchant type (%):")
print(merchant_conv)

# B2B vs B2C
b2b_conv = (
    df.groupby(['is_b2b','payment_method'])['converted']
    .mean().mul(100).round(1).unstack()
)
b2b_conv.index = ['B2C','B2B']
print("\nB2B vs B2C conversion (%):")
print(b2b_conv)

# Travel high-basket
travel_high = df[(df['merchant_type']=='travel') & (df['basket_value_gbp']>300)]
travel_conv = travel_high.groupby('payment_method')['converted'].agg(['mean','count'])
travel_conv['mean'] = (travel_conv['mean']*100).round(1)
print("\nTravel merchants, basket > £300:")
print(travel_conv.rename(columns={'mean':'conv_pct','count':'sessions'}))

# Cost per method for travel
travel_cost = (
    converted[converted['merchant_type']=='travel']
    .groupby('payment_method')[['effective_rate_pct','net_revenue_gbp','basket_value_gbp']]
    .mean().round(3)
)
print("\nCost for travel merchants:")
print(travel_cost)

print("""
Key insight:
  - B2B transactions convert BETTER on Pay by Bank (+8 pts vs B2C) — corporate buyers
    are comfortable with bank transfers and prefer avoiding card fees.
  - Travel high-basket: Pay by Bank conversion recovers vs overall because high-value
    users trust the bank flow more than entering card details for £300+ purchases.
  - Subscription merchants: cards dominate because recurring billing is frictionless;
    Pay by Bank Variable Recurring Payments (VRP) is the emerging alternative.
""")

---
## Task 7 — Build the Comparison Matrix

Synthesise your findings into a scored comparison matrix. Score each method **1–5** on each dimension (5 = best for merchant).

| Dimension | Weight | What to score |
|---|---|---|
| Conversion rate | 30% | Higher = better |
| Cost (fee rate) | 25% | Lower = better |
| Fraud / liability | 20% | Lower risk + bank liability = better |
| Settlement speed | 15% | Faster = better |
| UX friction | 10% | Fewer steps = better |

Calculate a weighted total score, then make recommendations for:
- **Scenario A:** Low-value subscription product (£10/month, high volume, mobile-first)
- **Scenario B:** High-value B2B invoice payment (£500–2000, desktop, CFO paying)

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────
# Build the matrix using your findings from Tasks 1-6
# Feel free to adjust scores based on your analysis


#### ✅ Answer — Task 7

In [ ]:
# Scores 1–5 per dimension (5 = best for merchant)
matrix = pd.DataFrame({
    'Conversion\n(30%)':     {'CARD': 3, 'PAY_BY_BANK': 2, 'PAYPAL': 4, 'APPLE_PAY': 5},
    'Cost\n(25%)':           {'CARD': 3, 'PAY_BY_BANK': 5, 'PAYPAL': 1, 'APPLE_PAY': 3},
    'Fraud/Liability\n(20%)':{'CARD': 3, 'PAY_BY_BANK': 5, 'PAYPAL': 1, 'APPLE_PAY': 4},
    'Settlement\n(15%)':     {'CARD': 2, 'PAY_BY_BANK': 5, 'PAYPAL': 3, 'APPLE_PAY': 2},
    'UX Friction\n(10%)':    {'CARD': 3, 'PAY_BY_BANK': 2, 'PAYPAL': 4, 'APPLE_PAY': 5},
})

weights = [0.30, 0.25, 0.20, 0.15, 0.10]
matrix['Weighted Score'] = (matrix * weights).sum(axis=1).round(2)
matrix = matrix.sort_values('Weighted Score', ascending=False)

print("Comparison Matrix (1–5, higher = better for merchant):")
print(matrix.to_string())

# Visualise
dims = [c for c in matrix.columns if c != 'Weighted Score']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
import matplotlib.colors as mcolors
score_data = matrix[dims].values
im = axes[0].imshow(score_data, cmap='RdYlGn', vmin=1, vmax=5, aspect='auto')
axes[0].set_xticks(range(len(dims)))
axes[0].set_xticklabels(dims, fontsize=8)
axes[0].set_yticks(range(len(matrix)))
axes[0].set_yticklabels(matrix.index)
axes[0].set_title('Comparison Matrix Heatmap')
for i in range(len(matrix)):
    for j in range(len(dims)):
        axes[0].text(j, i, str(score_data[i, j]), ha='center', va='center', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Weighted scores
methods_sorted = matrix.index.tolist()
scores = matrix['Weighted Score'].tolist()
bars = axes[1].barh(methods_sorted, scores, color=[COLORS[m] for m in methods_sorted])
axes[1].set_xlabel('Weighted Score')
axes[1].set_title('Overall Weighted Score')
axes[1].set_xlim(0, 5)
for bar, score in zip(bars, scores):
    axes[1].text(score + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{score:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("═" * 65)
print("SCENARIO A: Low-value subscription (£10/month, mobile-first)")
print("═" * 65)
print("""
Recommendation: APPLE PAY as default + CARD as fallback

Rationale:
  - Apple Pay: 1-tap biometric on mobile = highest conversion (~94%)
    At £10, fixed fees matter — Apple Pay fixed £0.20 + 1.55% = £0.355
  - PayPal not recommended: 2.49% rate on recurring = £0.549 per transaction,
    and buyer protection disputes on subscriptions are frequent.
  - Pay by Bank: VRP not yet widely deployed for recurring; standard PIS
    requires full re-authentication each month — worst UX for subscriptions.
  - Card via tokenised recurring (MIT) is viable fallback once initial consent given.
""")

print("═" * 65)
print("SCENARIO B: High-value B2B invoice (£500–2000, desktop, CFO)")
print("═" * 65)
print("""
Recommendation: PAY BY BANK as primary + CARD as fallback

Rationale:
  - Pay by Bank cost at £1,000: £0.08 flat = 0.008% effective rate.
    vs Card: 1.5–1.8% = £15–£18. Saving ~£15 per transaction.
  - B2B buyers convert well on Pay by Bank (+8 pts in data).
  - Zero chargeback risk — push payments cannot be reversed by payer.
  - Desktop UX for bank auth redirect is manageable for CFO use case.
  - T+0 settlement improves cash flow for high-value invoices.
  - Apple Pay not available on desktop; PayPal fee at £1,000 = £25.20 — prohibitive.
""")

---
## Task 8 — Daily Trends

Plot the **daily share of each payment method** over the 60-day period. Do you notice any trends?

Then calculate the **revenue at risk per day** — if Pay by Bank had 100% adoption (replacing all card transactions), how much would merchants save in fees per day? Plot this over time.

> This is a deliberately open task — there's no single right answer. Focus on what the data tells you.

In [ ]:
# ── YOUR CODE ───────────────────────────────────────────────────────────────


#### ✅ Answer — Task 8

In [ ]:
# Daily method share
daily_share = (
    df.groupby(['date','payment_method'])
    .size().unstack(fill_value=0)
    .apply(lambda r: r / r.sum() * 100, axis=1)
)

# Daily fee saving if PBB replaced all cards
daily_fees = (
    converted.groupby(['date','payment_method'])['fee_gbp']
    .sum().unstack(fill_value=0)
)
# PBB cost for same volume: £0.08 per transaction
daily_card_txns = converted[converted['payment_method']=='CARD'].groupby('date').size()
daily_pbb_cost  = daily_card_txns * 0.08
daily_card_cost = daily_fees.get('CARD', pd.Series(0, index=daily_fees.index))
daily_saving    = (daily_card_cost - daily_pbb_cost).clip(lower=0)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Stacked area: method share
daily_share.plot.area(ax=axes[0], color=list(COLORS.values()), alpha=0.8, linewidth=0)
axes[0].set_ylabel('Share (%)')
axes[0].set_title('Daily Payment Method Share')
axes[0].set_xlabel('')
axes[0].legend(loc='upper left', fontsize=9)

# Fee saving
axes[1].fill_between(daily_saving.index, daily_saving.values, color=C.teal if 'C' in dir() else '#0D9488', alpha=0.7)
axes[1].plot(daily_saving.index, daily_saving.rolling(7).mean(), color='#0D9488', linewidth=2, label='7-day rolling avg')
axes[1].set_ylabel('Potential daily saving (£)')
axes[1].set_title('Fee Saving if Pay by Bank Replaced All Card Transactions')
axes[1].legend()

plt.tight_layout()
plt.show()

total_card_fees = daily_fees.get('CARD', pd.Series()).sum()
total_pbb_fees  = daily_pbb_cost.sum()
total_saving    = total_card_fees - total_pbb_fees
annualised      = total_saving * 365 / 60

print(f"60-day card fees paid       : £{total_card_fees:,.0f}")
print(f"60-day PBB cost (same vol.) : £{total_pbb_fees:,.0f}")
print(f"60-day potential saving     : £{total_saving:,.0f}")
print(f"Annualised potential saving : £{annualised:,.0f}")
print("""
This illustrates why merchants are investing in Pay by Bank adoption.
The barrier is not cost — it's conversion rate and lack of consumer familiarity.
The industry question is: at what adoption level does the conversion gap
get offset by the fee saving? For high-value B2B that point is already here.
""")